In [1]:
from workflow_agent import LlmHandler

llm_handler = LlmHandler(
    llm_h_type="ollama",
    llm_h_params={
        "connection_string": "http://localhost:11434",
        "model_name": "gpt-oss:20b" #"llama3.1:latest" # "gemma3:27b"
    }
)

In [2]:
from workflow_agent import create_function_item

from typing import Type
from pydantic import BaseModel, Field

# --- Example usage ---

# Define mock functions and their associated Pydantic models:

# 1. get_weather function
def get_weather(city: str):
    """Get the current weather for a city from weather forcast api."""
    pass

class GetWeatherInput(BaseModel):
    city: str = Field(..., description="Name of the city for which weather to be extracted.")

class GetWeatherOutput(BaseModel):
    condition: str = Field(..., description="Weather condition in the requested city.")
    temperature: float = Field(..., description="Termperature in C in the requested city.")
    humidity: float = Field(None, description="Name of the city for which weather to be extracted.")

# 2. send_report_email function
def send_report_email(city: str, information: list):
    """Sends a report email with given information points to a city."""
    pass

class EmailInformationPoint(BaseModel):
    title: str = Field(None, description="Few word description of the information.")
    content: str = Field(..., description="Content of the information.")

class SendReportEmailInput(BaseModel):
    city: str = Field(..., description="Name of the city where report will be send.")
    information: list[EmailInformationPoint]

class SendReportEmailOutput(BaseModel):
    email_sent: bool = Field(..., description="Conformation that email was send successfully.")
    message: str = Field(None, description="Optional comments from the process.")

# 3. query_database function
def query_database(topic: str, location: str = None, uid: str = None):
    """Get information from the database with provided filters."""
    pass

class QueryDatabaseInput(BaseModel):
    topic: str = Field(..., description="Topic of a requested piece of information.")
    location: str = Field(None, description="Filter for location name.")
    uid: str = Field(None, description="Filter for unique indentifier of the database item.")

class QueryDatabaseOutput(BaseModel):
    info: str = Field(..., description="Content of the information.")
    uid: str = Field(None, description="Unique indentifier of the database item.")


# Create structured items for each function
available_functions = [
    create_function_item(get_weather, GetWeatherInput, GetWeatherOutput),
    create_function_item(send_report_email, SendReportEmailInput, SendReportEmailOutput),
    create_function_item(query_database, QueryDatabaseInput, QueryDatabaseOutput)
]

In [3]:
from workflow_agent import WorkflowPlanner

wp = WorkflowPlanner(llm_h = llm_handler)

In [4]:
task_description = """Query database to find information on birds and get latest weather for Berlin, then send an email there."""

workflow = await wp.generate_workflow(
    task_description=task_description, 
    available_functions=available_functions)

workflow

[{'name': 'query_database', 'args': {'topic': 'birds'}},
 {'name': 'get_weather', 'args': {'city': 'Berlin'}},
 {'name': 'send_report_email',
  'args': {'city': 'Berlin',
   'information': [{'title': 'Birds Information',
     'content': 'source: query_database.output.info'},
    {'title': 'Current Weather',
     'content': 'source: get_weather.output.condition'}]}}]

In [5]:
from workflow_agent import WorkflowAdaptor
from workflow_agent import InputCollector

wa = WorkflowAdaptor(llm_h = llm_handler, input_collector_class = InputCollector)
wa._initialize_input_collector_h()

In [ ]:
adapted_workflow = await wa.adapt_workflow(
    workflow=workflow, 
    available_functions=available_functions)

adapted_workflow